In [1]:
import os
import random
import time
import numpy as np

In [2]:
def load_documents():
    docs = {}
    for name in ["D1", "D2", "D3", "D4"]:
        with open(f"data/{name}.txt", "r", encoding="utf-8") as f:
            docs[name] = f.read()
    return docs

documents = load_documents()

for name, text in documents.items():
    print(name,"file has", "length:", len(text))

D1 file has length: 1749
D2 file has length: 1747
D3 file has length: 2132
D4 file has length: 1435


In [3]:
def char_kgrams(text, k):
    return set(text[i:i+k] for i in range(len(text)-k+1))

def word_kgrams(text, k):
    words = text.split()
    return set(" ".join(words[i:i+k]) for i in range(len(words)-k+1))

def jaccard(a, b):
    return len(a & b) / len(a | b)

In [5]:
doc_names = list(documents.keys())
pairs = [(doc_names[i], doc_names[j]) 
         for i in range(len(doc_names)) 
         for j in range(i+1, len(doc_names))]
print("************** Question 1 Create k-Gram ***************")
print("=== Character 2-grams ===")
char2 = {name: char_kgrams(text, 2) for name, text in documents.items()}
for a, b in pairs:
    print(a, b,"Jaccard Value for 2-grams char: ", jaccard(char2[a], char2[b]))

print("\n=== Character 3-grams ===")
char3 = {name: char_kgrams(text, 3) for name, text in documents.items()}
for a, b in pairs:
    print(a, b, "Jaccard Value for 3-gram char: ",jaccard(char3[a], char3[b]))

print("\n=== Word 2-grams ===")
word2 = {name: word_kgrams(text, 2) for name, text in documents.items()}
for a, b in pairs:
    print(a, b,"Jaccard Value for word 2-gram: ", jaccard(word2[a], word2[b]))

************** Question 1 Create k-Gram ***************
=== Character 2-grams ===
D1 D2 Jaccard Value for 2-grams char:  0.9811320754716981
D1 D3 Jaccard Value for 2-grams char:  0.8156996587030717
D1 D4 Jaccard Value for 2-grams char:  0.6444444444444445
D2 D3 Jaccard Value for 2-grams char:  0.8
D2 D4 Jaccard Value for 2-grams char:  0.6412698412698413
D3 D4 Jaccard Value for 2-grams char:  0.6529968454258676

=== Character 3-grams ===
D1 D2 Jaccard Value for 3-gram char:  0.977979274611399
D1 D3 Jaccard Value for 3-gram char:  0.5803571428571429
D1 D4 Jaccard Value for 3-gram char:  0.3050847457627119
D2 D3 Jaccard Value for 3-gram char:  0.5680473372781065
D2 D4 Jaccard Value for 3-gram char:  0.30590339892665475
D3 D4 Jaccard Value for 3-gram char:  0.31212381771281167

=== Word 2-grams ===
D1 D2 Jaccard Value for word 2-gram:  0.9407665505226481
D1 D3 Jaccard Value for word 2-gram:  0.18234165067178504
D1 D4 Jaccard Value for word 2-gram:  0.03024193548387097
D2 D3 Jaccard Value 

In [6]:
def generate_hash_functions(t, m):
    hash_funcs = []
    for _ in range(t):
        a = random.randint(1, m-1)
        b = random.randint(0, m-1)
        hash_funcs.append((a, b))
    return hash_funcs

def minhash_signature(kgram_set, hash_funcs, m):
    signature = []
    for a, b in hash_funcs:
        min_val = min(((a * hash(x) + b) % m) for x in kgram_set)
        signature.append(min_val)
    return signature

def signature_similarity(sig1, sig2):
    return sum(1 for i in range(len(sig1)) if sig1[i] == sig2[i]) / len(sig1)

In [7]:
m = 20000
d1_set = char3["D1"]
d2_set = char3["D2"]

exact_similarity = jaccard(d1_set, d2_set)
print("*********** Question 2 Min-Hashing ************")
print("Exact similarity:", exact_similarity)

for t in [20, 60, 150, 300, 600]:
    hash_funcs = generate_hash_functions(t, m)
    sig1 = minhash_signature(d1_set, hash_funcs, m)
    sig2 = minhash_signature(d2_set, hash_funcs, m)
    est = signature_similarity(sig1, sig2)
    print("t =", t, "Estimated similarity:", est)

*********** Question 2 Min-Hashing ************
Exact similarity: 0.977979274611399
t = 20 Estimated similarity: 1.0
t = 60 Estimated similarity: 0.9833333333333333
t = 150 Estimated similarity: 0.9933333333333333
t = 300 Estimated similarity: 0.9666666666666667
t = 600 Estimated similarity: 0.9733333333333334


In [8]:
def lsh_probability(s, r, b):
    return 1 - (1 - s**r)**b

r = 8
b = 20
print("****************** Question 3 LSH Probabilities ************")
print("Using r =", r, "b =", b)

for a, b_pair in pairs:
    s = jaccard(char3[a], char3[b_pair])
    prob = lsh_probability(s, r, b)
    print(a,"&", b_pair, "Probability:", prob)

****************** Question 3 LSH Probabilities ************
Using r = 8 b = 20
D1 & D2 Probability: 0.9999999999999998
D1 & D3 Probability: 0.22822420166613133
D1 & D4 Probability: 0.001499976039931461
D2 & D3 Probability: 0.1958804369207785
D2 & D4 Probability: 0.0015324562526581875
D3 & D4 Probability: 0.0018000048667899948


In [10]:
import urllib.request
import zipfile

url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
filename = "ml-100k.zip"

urllib.request.urlretrieve(url, filename)

with zipfile.ZipFile(filename, 'r') as zip_ref:
    zip_ref.extractall("data")

In [12]:
import pandas as pd

data = pd.read_csv("data/ml-100k/u.data", sep="\t",
                   names=["user","movie","rating","time"])

user_movies = data.groupby("user")["movie"].apply(set)
user_sets = list(user_movies.values)

from itertools import combinations

def jaccard(A,B):
    return len(A&B)/len(A|B)

exact=[]
for i,j in combinations(range(len(user_sets)),2):
    if jaccard(user_sets[i],user_sets[j])>=0.5:
        exact.append((i+1,j+1))
print("User pairs with similarity >= 0.5:")
for pair in exact:
    print(pair)

print("Total number of such pairs:", len(exact))

User pairs with similarity >= 0.5:
(197, 600)
(197, 826)
(328, 788)
(408, 898)
(451, 489)
(489, 587)
(554, 764)
(600, 826)
(674, 879)
(800, 879)
Total number of such pairs: 10


In [13]:
all_movies = list(set().union(*user_sets))
idx = {m:i for i,m in enumerate(all_movies)}

def minhash_users(num_hash):
    sig = np.full((num_hash,len(user_sets)),np.inf)
    for i in range(num_hash):
        a,b = np.random.randint(1,10000), np.random.randint(0,10000)
        for col,mset in enumerate(user_sets):
            for m in mset:
                x = idx[m]
                h = (a*x+b)%10007
                sig[i,col] = min(sig[i,col],h)
    return sig

def approx(sig,i,j):
    return np.mean(sig[:,i]==sig[:,j])

for t in [50,100,200]:
    sig = minhash_users(t)
    count=0
    for i,j in combinations(range(len(user_sets)),2):
        if approx(sig,i,j)>=0.5:
            count+=1
    print(f"t={t}, pairs ≥0.5:", count)

t=50, pairs ≥0.5: 137
t=100, pairs ≥0.5: 51
t=200, pairs ≥0.5: 11


In [14]:
from itertools import combinations

# Exact pairs with similarity ≥ threshold
def get_exact_pairs(threshold=0.5):
    exact = set()

    for i, j in combinations(range(len(user_sets)), 2):
        sim = jaccard(user_sets[i], user_sets[j])
        if sim >= threshold:
            exact.add((i, j))

    return exact

In [15]:
# Evaluate MinHash accuracy
def evaluate_minhash(t, threshold=0.5, runs=5):

    exact = get_exact_pairs(threshold)

    fp_total = 0
    fn_total = 0
    print(f"\n--- Running 5 trials for t={t} hash functions ---")
    for kk in range(runs):

        sig = minhash_users(t)

        approx_pairs = set()

        for i, j in combinations(range(len(user_sets)), 2):
            if approx(sig, i, j) >= threshold:
                approx_pairs.add((i, j))

        fp = len(approx_pairs - exact)
        fn = len(exact - approx_pairs)

        fp_total += fp
        fn_total += fn
        print(f"  Run {kk+1}/{runs} => False Positives: {fp}, False Negatives: {fn}")

    print("-------------------------------------")
    print(f"AVERAGE FALSE POSITVES for t = {t}:", fp_total / runs)
    print(f"AVERAGE FALSE NEGATIVES for t = {t}:", fn_total / runs)
    print("-------------------------------------")

In [16]:
print("*********** Question 4: Min-Hashing on MovieLens Dataset ***********")
for t in [50, 100, 200]:
    evaluate_minhash(t)

*********** Question 4: Min-Hashing on MovieLens Dataset ***********

--- Running 5 trials for t=50 hash functions ---
  Run 1/5 => False Positives: 72, False Negatives: 2
  Run 2/5 => False Positives: 221, False Negatives: 1
  Run 3/5 => False Positives: 31, False Negatives: 3
  Run 4/5 => False Positives: 130, False Negatives: 1
  Run 5/5 => False Positives: 330, False Negatives: 1
-------------------------------------
AVERAGE FALSE POSITVES for t = 50: 156.8
AVERAGE FALSE NEGATIVES for t = 50: 1.6
-------------------------------------

--- Running 5 trials for t=100 hash functions ---
  Run 1/5 => False Positives: 46, False Negatives: 1
  Run 2/5 => False Positives: 24, False Negatives: 4
  Run 3/5 => False Positives: 26, False Negatives: 2
  Run 4/5 => False Positives: 14, False Negatives: 2
  Run 5/5 => False Positives: 13, False Negatives: 5
-------------------------------------
AVERAGE FALSE POSITVES for t = 100: 24.6
AVERAGE FALSE NEGATIVES for t = 100: 2.8
--------------------

## Part 5 — LSH on MovieLens

In [24]:
def lsh(sig,r):
    bands = sig.shape[0]//r
    buckets={}
    for b in range(bands):
        band=sig[b*r:(b+1)*r]
        for col in range(sig.shape[1]):
            key=tuple(band[:,col])
            buckets.setdefault((b,key),[]).append(col)
    cand=set()
    for bucket in buckets.values():
        if len(bucket)>1:
            for i in range(len(bucket)):
                for j in range(i+1,len(bucket)):
                    cand.add((bucket[i]+1,bucket[j]+1))
    return cand

sig50 = minhash_users(50)
print("LSH candidates (50 hashes, r=5):", len(lsh(sig50,5)))

LSH candidates (50 hashes, r=5): 533


In [25]:
sig100 = minhash_users(100)

candidates_100 = lsh(sig100, r=5)

print("LSH candidates (100 hashes, r=5):", len(candidates_100))


LSH candidates (100 hashes, r=5): 1540


In [19]:
sig200 = minhash_users(200)

candidates_200_a = lsh(sig200, r=5)

print("LSH candidates (200 hashes, r=5):", len(candidates_200_a))

LSH candidates (200 hashes, r=5): 1544


In [20]:
candidates_200_b = lsh(sig200, r=10)

print("LSH candidates (200 hashes, r=10):", len(candidates_200_b))

LSH candidates (200 hashes, r=10): 3


In [21]:
# Exact pairs for given threshold
def exact_pairs_threshold(threshold):
    pairs = set()

    for i, j in combinations(range(len(user_sets)), 2):
        if jaccard(user_sets[i], user_sets[j]) >= threshold:
            pairs.add((i, j))

    return pairs


In [ ]:
results = []
def evaluate_lsh(sig, r, threshold,config_name, runs=5):

    exact = exact_pairs_threshold(threshold)
   
    fp_total = 0
    fn_total = 0

    for _ in range(runs):

        candidates = lsh(sig, r)

        approx = set((i-1, j-1) for (i, j) in candidates)

        fp = len(approx - exact)
        fn = len(exact - approx)

        fp_total += fp
        fn_total += fn

    print(f"\nThreshold = {threshold}, r = {r}")
    print("Avg False Positives:", fp_total / runs)
    print("Avg False Negatives:", fn_total / runs)
   
    results.append({
        "Configuration": config_name,
        "Threshold": threshold,
        "r": r,
        "Avg False Positives": fp_total / runs,
        "Avg False Negatives":  fn_total / runs
    })
   


In [32]:
print("********** Question 5 LSH on MovieLens Dataset **************")
# 50 hashes
sig50 = minhash_users(50)
evaluate_lsh(sig50, r=5, threshold=0.6, config_name="50 hashes")
evaluate_lsh(sig50, r=5, threshold=0.8, config_name="50 hashes")

# 100 hashes
sig100 = minhash_users(100)
evaluate_lsh(sig100, r=5, threshold=0.6, config_name="100 hashes")
evaluate_lsh(sig100, r=5, threshold=0.8, config_name="100 hashes")

# 200 hashes
sig200 = minhash_users(200)
evaluate_lsh(sig200, r=5, threshold=0.6, config_name="200 hashes")
evaluate_lsh(sig200, r=5, threshold=0.8, config_name="200 hashes")
evaluate_lsh(sig200, r=10, threshold=0.6, config_name="200 hashes")
evaluate_lsh(sig200, r=10, threshold=0.8, config_name="200 hashes")

********** Question 5 LSH on MovieLens Dataset **************

Threshold = 0.6, r = 5
Avg False Positives: 688.0
Avg False Negatives: 0.0

Threshold = 0.8, r = 5
Avg False Positives: 690.0
Avg False Negatives: 0.0

Threshold = 0.6, r = 5
Avg False Positives: 1107.0
Avg False Negatives: 0.0

Threshold = 0.8, r = 5
Avg False Positives: 1109.0
Avg False Negatives: 0.0

Threshold = 0.6, r = 5
Avg False Positives: 2312.0
Avg False Negatives: 0.0

Threshold = 0.8, r = 5
Avg False Positives: 2314.0
Avg False Negatives: 0.0

Threshold = 0.6, r = 10
Avg False Positives: 4.0
Avg False Negatives: 1.0

Threshold = 0.8, r = 10
Avg False Positives: 5.0
Avg False Negatives: 0.0


In [ ]:
import pandas as pd
df_results = pd.DataFrame(results)
df_results.sort_values(by=["Configuration", "Threshold"])

NameError: name 'df_results' is not defined